In [ ]:
from pathlib import Path
from zipfile import ZipFile
import subprocess, sys, torch
from google.colab import drive
drive.mount('/content/drive')
disk = Path('/content/drive/MyDrive')
work = Path('/content/tea_phase5')
work.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU runtime'
for expected, name in [('data/graph_expanded/pyg/snapshots.pt','tea_phase4_colab.zip'),
                       ('results/expanded/selected_test_predictions.csv','phase5_colab.zip')]:
    if not (work/expected).is_file():
        if (disk/'tea_phase5'/expected).is_file():
            import shutil
            shutil.copytree(disk/'tea_phase5', work, dirs_exist_ok=True)
        else:
            with ZipFile(disk/name) as z:
                z.extractall(work)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=work,check=True)
run('-m','pip','install','-q','torch_geometric==2.8.1')
path = work / 'src/eval/check_phase5.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Independently verify local or Colab Phase 5 outputs against Phase 3."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nROOT = Path(__file__).resolve().parents[2]\nKEY = ["target", "graph", "seed"]\nID = {"campus_id": str}\n\n\ndef check(directory, snapshot_path, phase3, require_cuda=False):\n    protocol = json.loads((directory / "protocol.json").read_text())\n    if require_cuda and protocol["device"] != "cuda":\n        raise ValueError("Expected a CUDA run")\n    with snapshot_path.open("rb") as f:\n        assert hashlib.file_digest(f, "sha256").hexdigest() == protocol["snapshot_sha256"]\n    data = torch.load(snapshot_path, weights_only=False, map_location="cpu")\n    assert data["targets"] == protocol["targets"] and data["split"] == protocol["split"]\n    test = data["snapshots"][-1]\n    selected = pd.DataFrame(json.loads((directory / "selection_locked.json").read_text()))\n    grid = pd.read_csv(directory / "validation_grid.csv")\n    pred = pd.read_csv(directory / "predictions.csv", dtype={**ID, "y_pred": np.float32})\n    ref = pd.read_csv(phase3 / "selected_test_predictions.csv", dtype=ID)\n    ref = ref.merge(pd.read_csv(phase3 / "feature_selection_locked.csv")[["target","input_set"]], validate="many_to_one")\n    assert pred.year.eq(data["years"][-1]).all() and pred.split.eq("test").all()\n    assert set(pred.model) == {protocol["model"]}\n    assert not pred.duplicated(KEY + ["campus_id"]).any()\n    scores = []\n    for j, target in enumerate(data["targets"]):\n        g = grid[grid.target == target]\n        eligible = {"combined","district","region","geographic","demographic"}\n        if target in data["academic_relation"]:\n            eligible.add(data["academic_relation"][target])\n        assert set(g.graph) == eligible and len(g) == len(eligible)\n        single = g[g.graph != "combined"].sort_values("val_mae", kind="stable").iloc[0]\n        chosen = selected[selected.target == target]\n        assert set(chosen.graph) == {"combined", single.graph} and len(chosen) == 2\n        mask = test["campus"].eval_mask[:, j]\n        ids = np.array(data["campus_ids"])[mask.numpy()]\n        previous = test["campus"].y_lag1[mask, j].numpy()\n        for row in chosen.itertuples():\n            valrow = g[g.graph == row.graph].iloc[0]\n            assert row.epochs == valrow.epochs and np.isclose(row.val_mae, valrow.val_mae)\n            for seed in data["split"]["seeds"]:\n                rows = pred[(pred.target == target) & (pred.graph == row.graph) & (pred.seed == seed)].set_index("campus_id")\n                benchmark = ref[(ref.target == target) & (ref.seed == seed)].set_index("campus_id")\n                assert set(rows.index) == set(ids) == set(benchmark.index)\n                truth = benchmark.loc[ids].y_true.to_numpy()\n                np.testing.assert_allclose(rows.loc[ids].y_true, truth, rtol=0, atol=1e-10)\n                error = rows.loc[ids].y_pred.to_numpy() - truth\n                assert np.isfinite(error).all()\n                mae = np.abs(error).mean()\n                scores.append(dict(target=target,graph=row.graph,seed=seed,n=len(ids),mae=mae,\n                    rmse=np.mean(error**2)**0.5,r2=1-np.sum(error**2)/np.sum((truth-truth.mean())**2),\n                    coverage_pct=100*len(ids)/int(test["campus"].present.sum()),\n                    delta_vs_phase3=mae-np.abs(benchmark.loc[ids].y_pred-truth).mean(),\n                    delta_vs_persistence=mae-np.abs(previous-truth).mean()))\n    calculated = pd.DataFrame(scores).set_index(KEY).sort_index()\n    stored = pd.read_csv(directory / "per_seed.csv").set_index(KEY).sort_index()\n    assert calculated.index.equals(stored.index) and len(pred) == calculated.n.sum()\n    np.testing.assert_allclose(stored[calculated.columns], calculated, atol=1e-8)\n    mean = calculated.groupby(level=["target","graph"]).mean()\n    mean["mae_sd"] = calculated.mae.groupby(level=["target","graph"]).std(ddof=0)\n    comparison = pd.read_csv(directory / "comparison.csv").set_index(["target","graph"]).sort_index()\n    assert mean.index.equals(comparison.index)\n    np.testing.assert_allclose(comparison[mean.columns], mean, atol=1e-8)\n    print(f"Verified {protocol[\'model\']} on {protocol[\'device\']}: {len(pred):,} predictions, {len(stored)} seed comparisons",flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("directories", nargs="+", type=Path)\n    parser.add_argument("--require-cuda",action="store_true")\n    args = parser.parse_args()\n    for directory in args.directories:\n        check(directory, ROOT/"data/graph_expanded/pyg/snapshots.pt", ROOT/"results/expanded", args.require_cuda)\n')
path = work / 'src/models/phase5_cells.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Temporal graph baselines for annual, weighted campus snapshots."""\nimport torch\nfrom torch import nn\n\n\nclass TGCN(nn.Module):\n    def __init__(self, features, hidden, attention=False):\n        super().__init__()\n        self.hidden = hidden\n        self.project = nn.Linear(features, 3 * hidden)\n        self.gates = nn.ModuleList(nn.Linear(2 * hidden, hidden) for _ in range(3))\n        self.attention = nn.Parameter(torch.zeros(3)) if attention else None\n        self.readout = nn.Linear(hidden, 1)\n\n    def forward(self, aggregated):\n        projected = self.project(aggregated)\n        state = projected.new_zeros(projected.shape[1], self.hidden)\n        states = []\n        for frame in projected:\n            z, r, c = frame.chunk(3, dim=-1)\n            prior = state if self.attention is None else torch.zeros_like(state)\n            update = self.gates[0](torch.cat((z, prior), -1)).sigmoid()\n            reset = self.gates[1](torch.cat((r, prior), -1)).sigmoid()\n            proposal = self.gates[2](torch.cat((c, reset * prior), -1)).tanh()\n            state = update * prior + (1 - update) * proposal\n            states.append(state)\n        sequence = torch.stack(states)\n        if self.attention is not None:\n            # Prefix-only, sliding three-period A3TGCN window.\n            outputs = []\n            for end in range(1, len(states) + 1):\n                window = sequence[max(0, end-3):end]\n                weights = self.attention[-len(window):].softmax(0)\n                outputs.append((window * weights[:, None, None]).sum(0))\n            sequence = torch.stack(outputs)\n        return self.readout(sequence).squeeze(-1)\n\n\nclass GraphWindow:\n    def __init__(self, x, supports):\n        self.x = x\n        self.supports = supports\n        self.shape = x.shape\n\n    def __getitem__(self, key):\n        return GraphWindow(self.x[key], self.supports[key])\n\n    def to(self, device):\n        return GraphWindow(self.x.to(device), [a.to(device) for a in self.supports])\n\n\nclass GConvGRU(nn.Module):\n    """Two-term Chebyshev filters for inputs and hidden states (lambda_max=2)."""\n    def __init__(self, features, hidden):\n        super().__init__()\n        self.hidden = hidden\n        self.input = nn.Linear(2 * features, 3 * hidden)\n        self.gates = nn.Linear(2 * hidden, 2 * hidden)\n        self.candidate = nn.Linear(2 * hidden, hidden)\n        self.readout = nn.Linear(hidden, 1)\n\n    def forward(self, window):\n        state = window.x.new_zeros(window.x.size(1), self.hidden)\n        outputs = []\n        for x, adjacency in zip(window.x, window.supports):\n            z, r, c = self.input(x).chunk(3, -1)\n            hz, hr = self.gates(torch.cat((state, -adjacency @ state), -1)).chunk(2, -1)\n            update, reset = (z+hz).sigmoid(), (r+hr).sigmoid()\n            gated = reset * state\n            candidate = (c + self.candidate(torch.cat((gated, -adjacency @ gated), -1))).tanh()\n            state = update * state + (1-update) * candidate\n            outputs.append(self.readout(state).squeeze(-1))\n        return torch.stack(outputs)\n\n\nclass GraphWaveNet(nn.Module):\n    """Causal dilated gated blocks with residual, skip, and graph diffusion."""\n    def __init__(self, features, hidden):\n        super().__init__()\n        self.start = nn.Linear(features, hidden)\n        self.filters = nn.ModuleList(nn.Linear(2*hidden, 2*hidden) for _ in range(2))\n        self.diffuse = nn.ModuleList(nn.Linear(3*hidden, hidden) for _ in range(2))\n        self.skip = nn.ModuleList(nn.Linear(hidden, hidden) for _ in range(2))\n        self.end = nn.Sequential(nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))\n\n    def forward(self, window):\n        sequence = self.start(window.x)\n        skip = 0\n        for dilation, gate, diffusion, project in zip((1, 2), self.filters, self.diffuse, self.skip):\n            delayed = torch.cat((torch.zeros_like(sequence[:dilation]), sequence[:-dilation]), 0)\n            f, g = gate(torch.cat((sequence, delayed), -1)).chunk(2, -1)\n            gated = f.tanh() * g.sigmoid()\n            skip = skip + project(gated)\n            first = torch.stack([a @ x for a, x in zip(window.supports, gated)])\n            second = torch.stack([a @ x for a, x in zip(window.supports, first)])\n            sequence = sequence + diffusion(torch.cat((gated, first, second), -1))\n        # Include final diffusion to retain both spatial blocks in the readout.\n        return self.end(skip + sequence).squeeze(-1)\n')
path = work / 'src/models/phase5_gcn_lstm.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Phase 5 GCN + LSTM benchmark on the verified expanded snapshot."""\nimport argparse\nimport hashlib\nimport json\nimport io\nfrom zipfile import ZipFile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch_geometric.nn.conv.gcn_conv import gcn_norm\n\nROOT = Path(__file__).resolve().parents[2]\nSNAP = ROOT / "data/graph_expanded/pyg/snapshots.pt"\nBASE = ROOT / "results/expanded"\nLAYERS = ["district", "geographic", "demographic", "region"]\nEPOCHS = 30\nHIDDEN = 32\nLR = 0.003\n\n\nclass Model(nn.Module):\n    def __init__(self, features):\n        super().__init__()\n        self.gcn = nn.Linear(features, HIDDEN)\n        self.lstm = nn.LSTM(HIDDEN, HIDDEN)\n        self.head = nn.Linear(HIDDEN, 1)\n\n    def forward(self, aggregated):\n        encoded = self.gcn(aggregated).relu()\n        sequence, _ = self.lstm(encoded)\n        return self.head(sequence).squeeze(-1)\n\n\ndef graph_inputs(snapshot, layers, family="gcn_lstm"):\n    # Sum overlapping relation weights before adding one self-loop per campus.\n    stores = [snapshot["campus", layer, "campus"] for layer in layers]\n    x = snapshot["campus"].x\n    edges = torch.cat([store.edge_index for store in stores], dim=1)\n    weights = torch.cat([store.edge_weight for store in stores])\n    if family == "graphwavenet":\n        degree = weights.new_zeros(x.size(0)).index_add_(0, edges[0], weights)\n        weights = weights / degree[edges[0]].clamp_min(1e-12)\n    else:\n        edges, weights = gcn_norm(edges, weights, x.size(0), add_self_loops=family != "gconvgru")\n    adjacency = torch.sparse_coo_tensor(edges.flip(0), weights, (len(x), len(x))).coalesce()\n    # A_norm X is independent of trainable weights. Projecting it in Model\n    # computes the same convolution as A_norm X W + bias at every epoch.\n    support = adjacency.to_sparse_csr()\n    if family == "gconvgru":\n        return torch.cat((x, -(support @ x)), -1), support\n    if family == "graphwavenet":\n        return x, support\n    return support @ x\n\n\ndef fit(x, labels, eligible, stop, seed, device, epochs, tune, family="gcn_lstm"):\n    torch.manual_seed(seed)\n    from phase5_cells import TGCN, GConvGRU, GraphWaveNet\n    constructors = {"gcn_lstm": Model, "tgcn": lambda f: TGCN(f, HIDDEN),\n                    "a3tgcn": lambda f: TGCN(f, HIDDEN, attention=True),\n                    "gconvgru": lambda f: GConvGRU(f//2, HIDDEN),\n                    "graphwavenet": lambda f: GraphWaveNet(f, HIDDEN)}\n    model = constructors[family](x.shape[-1]).to(device)\n    opt = torch.optim.Adam(model.parameters(), lr=LR)\n    history = labels[1:stop]\n    center, scale = history[history.isfinite()].mean(), history[history.isfinite()].std()\n    observed = history.isfinite()\n    best = (float("inf"), 0)\n    for epoch in range(1, epochs + 1):\n        pred = model(x[:stop])[1:]\n        loss = ((pred[observed] - (history[observed] - center) / scale) ** 2).mean()\n        opt.zero_grad()\n        loss.backward()\n        opt.step()\n        if tune:\n            with torch.no_grad():\n                forecast = model(x[:stop + 1])[-1] * scale + center\n                error = (forecast[eligible[stop]] - labels[stop, eligible[stop]]).abs().mean().item()\n                best = min(best, (error, epoch))\n    with torch.no_grad():\n        prediction = model(x[:stop + 1])[-1] * scale + center\n    return best, prediction.cpu().numpy(), model, center.item(), scale.item()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--device", choices=["cuda", "cpu"], required=True)\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    parser.add_argument("--model", choices=["gcn_lstm","tgcn","gconvgru","a3tgcn","graphwavenet"], default="gcn_lstm")\n    args = parser.parse_args()\n    model_name = {"tgcn":"T-GCN","gconvgru":"GConvGRU","a3tgcn":"A3TGCN","graphwavenet":"GraphWaveNet (fixed supports)","gcn_lstm":"GCN + LSTM"}[args.model]\n    torch.set_num_threads(2)\n    out = args.out / ("smoke" if args.smoke else "benchmark")\n    out.mkdir(parents=True, exist_ok=True)\n    data = torch.load(SNAP, weights_only=False, map_location="cpu")\n    snaps = data["snapshots"]\n    split = data["split"]\n    assert data["years"] == split["train_years"] + split["val_years"] + split["test_years"]\n    val = len(split["train_years"])\n    targets = ["graduation_4yr"] if args.smoke else data["targets"]\n    epochs = 2 if args.smoke else EPOCHS\n    seeds = [0] if args.smoke else split["seeds"]\n    protocol = dict(model=model_name, hidden=HIDDEN, learning_rate=LR, epochs=epochs,\n                    validation_seed=0, split=split, device=args.device, torch_version=torch.__version__,\n                    snapshot_sha256=hashlib.sha256(SNAP.read_bytes()).hexdigest(), targets=targets)\n    (out / "protocol.json").write_text(json.dumps(protocol, indent=2))\n    cache = {}\n\n    def features(layers):\n        key = tuple(layers)\n        if key not in cache:\n            frames = [graph_inputs(s, layers, args.model) for s in snaps]\n            if args.model in ("graphwavenet","gconvgru"):\n                from phase5_cells import GraphWindow\n                values, supports = zip(*frames)\n                cache[key] = GraphWindow(torch.stack(values), list(supports))\n            else:\n                cache[key] = torch.stack(frames)\n        return cache[key].to(args.device)\n\n    def outcomes(target):\n        j = data["targets"].index(target)\n        return [torch.stack([s["campus"][key][:, j] for s in snaps]).to(args.device)\n                for key in ("y", "eval_mask")]\n\n    choices = []\n    tuning = []\n    for target in targets:\n        layers = LAYERS.copy()\n        if target in data["academic_relation"]:\n            layers.append(data["academic_relation"][target])\n        labels, masks = outcomes(target)\n        candidates = []\n        for name, relations in [("combined", layers)] + [(layer, [layer]) for layer in layers]:\n            best, *_ = fit(features(relations), labels, masks, val, 0, args.device, epochs, True, args.model)\n            row = dict(target=target, graph=name, relations=relations, val_mae=best[0], epochs=best[1])\n            assert np.isfinite(best[0]) and best[1] > 0\n            candidates.append(row)\n        choices.extend([candidates[0], min(candidates[1:], key=lambda r: r["val_mae"])])\n        tuning.extend(candidates)\n        print("Validation complete:", target, flush=True)\n    pd.DataFrame(tuning).to_csv(out / "validation_grid.csv", index=False)\n    (out / "selection_locked.json").write_text(json.dumps(choices, indent=2))\n    if args.smoke:\n        print("Smoke validation passed", flush=True)\n        return\n    reference = pd.read_csv(BASE / "selected_test_predictions.csv", dtype={"campus_id": str})\n    locked = pd.read_csv(BASE / "feature_selection_locked.csv")[["target", "input_set"]]\n    reference = reference.merge(locked, on=["target", "input_set"], validate="many_to_one")\n    predictions, scores = [], []\n    archive_path = out / "checkpoints.partial.zip"\n    with ZipFile(archive_path, "w"):\n        pass\n    for row in choices:\n        target = row["target"]\n        j = data["targets"].index(target)\n        labels, masks = outcomes(target)\n        mask = masks[-1].cpu().numpy()\n        ids = np.array(data["campus_ids"])[mask]\n        for seed in seeds:\n            _, pred, model, center, scale = fit(features(row["relations"]), labels, masks, val+1, seed, args.device, row["epochs"], False, args.model)\n            truth = labels[-1].cpu().numpy()[mask]\n            ref = reference[(reference.target == target) & (reference.seed == seed)].set_index("campus_id")\n            assert ref.index.is_unique and set(ref.index) == set(ids)\n            ref = ref.loc[ids]\n            np.testing.assert_allclose(ref.y_true, truth, atol=1e-5)\n            assert np.isfinite(pred[mask]).all()\n            err = pred[mask] - ref.y_true.to_numpy()\n            mae = np.abs(err).mean()\n            persist = snaps[-1]["campus"].y_lag1[:, j].numpy()[mask]\n            scores.append(dict(target=target, graph=row["graph"], seed=seed, n=len(ids),\n                               coverage_pct=100*len(ids)/int(snaps[-1]["campus"].present.sum()),\n                               mae=mae, rmse=np.sqrt(np.mean(err**2)),\n                               r2=1-np.sum(err**2)/np.sum((ref.y_true-ref.y_true.mean())**2),\n                               delta_vs_phase3=mae-(ref.y_pred-ref.y_true).abs().mean(),\n                               delta_vs_persistence=mae-np.abs(persist-ref.y_true).mean()))\n            predictions.append(pd.DataFrame(dict(model=model_name, graph=row["graph"], target=target, seed=seed,\n                campus_id=ids, year=data["years"][-1], split="test", y_true=ref.y_true.to_numpy(), y_pred=pred[mask])))\n            buffer = io.BytesIO()\n            torch.save(dict(state_dict=model.state_dict(), center=center, scale=scale, selection=row), buffer)\n            with ZipFile(archive_path, "a") as archive:\n                archive.writestr(f"{target}_{row[\'graph\']}_{seed}.pt", buffer.getvalue())\n    archive_path.replace(out / "checkpoints.zip")\n    pd.concat(predictions).to_csv(out/"predictions.csv", index=False)\n    metrics = pd.DataFrame(scores)\n    metrics.to_csv(out/"per_seed.csv", index=False)\n    grouped = metrics.groupby(["target","graph"])\n    grouped.mean(numeric_only=True).drop(columns="seed").assign(mae_sd=grouped.mae.std(ddof=0)).to_csv(out/"comparison.csv")\n    print("Completed", len(choices), "fixed target/graph comparisons", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
# Audit your existing GPU run. The checker reads the original campus predictions.
existing = disk/'tea_phase5_gcn_lstm/benchmark'
run('src/eval/check_phase5.py','--require-cuda',existing)

In [ ]:
# Each completed family can be independently verified and reused after a disconnect.
for family in ['tgcn','a3tgcn','gconvgru','graphwavenet']:
    result = disk/'tea_phase5_results'/family
    report = result/'benchmark/comparison.csv'
    if not report.exists():
        run('-u','src/models/phase5_gcn_lstm.py','--model',family,'--device','cuda','--out',result)
    run('src/eval/check_phase5.py','--require-cuda',result/'benchmark')

In [ ]:
import pandas as pd
parts = []
for family in ['gcn_lstm','tgcn','a3tgcn','gconvgru','graphwavenet']:
    folder = existing if family == 'gcn_lstm' else disk/'tea_phase5_results'/family/'benchmark'
    parts.append(pd.read_csv(folder/'comparison.csv').assign(model=family))
combined = pd.concat(parts,ignore_index=True)
combined.to_csv(disk/'phase5_scores.csv',index=False)
display(combined[combined.graph.eq('combined')].pivot(index='target',columns='model',values='mae').round(3))
print('All GPU scores saved in My Drive → phase5_scores.csv')
